# Fine-Tuning Chatterbox for Maltese Language Support

This notebook trains the Chatterbox multilingual TTS model for Maltese language support using the MASRI_HEADSET_v2 dataset.

**Key Features:**
- Optimized for Google Colab free tier (T4 GPU, 15GB RAM)
- Trains text embeddings and T3 transformer only
- Prevents catastrophic forgetting with mixed-language training
- Uses gradient accumulation for effective larger batch sizes

**Dataset:** [Bluefir/MASRI_HEADSET_v2](https://huggingface.co/datasets/Bluefir/MASRI_HEADSET_v2)

**Training Strategy:**
- 40% Maltese (from MASRI dataset)
- 35% Arabic (preserve Semitic knowledge)
- 20% Italian (preserve Romance knowledge)
- 5% English (maintain general performance)


## 1. Setup and Installation

### Requirements
- Python 3.12 kernel

In [8]:
# Check GPU availability
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

/bin/bash: line 1: nvidia-smi: command not found
PyTorch version: 2.6.0+cu124
CUDA available: False


In [9]:
import os
import shutil

# 1. CLEANUP: Remove existing chatterbox folder to fix the "nested folder" issue
print("Cleaning up old files...")
if os.path.exists("/content/chatterbox"):
    shutil.rmtree("/content/chatterbox")

# 2. CLONE: Clone freshly into /content
%cd /content
if not os.path.exists('/content/chatterbox'):
    print("Cloning repository...")
    !git clone https://github.com/Wubpooz/chatterbox.git
    %cd chatterbox
    !git checkout copilot/add-maltese-language-support
    print("✓ Repository cloned and branch checked out")
else:
    %cd chatterbox
    print("✓ Repository already exists")

# 3. PATCH: Fix the configuration files BEFORE installing
print("\nPatching configuration files...")

# Fix 1: Relax NumPy version in pyproject.toml for Python 3.12
!sed -i 's/numpy>=1.24.0,<1.26.0/numpy>=1.24.0,<2.0.0/' pyproject.toml
print("✓ Patched pyproject.toml")

# Fix 2: Repair the broken import in s3tokenizer.py
target_file = "src/chatterbox/models/s3tokenizer/s3tokenizer.py"
if os.path.exists(target_file):
    with open(target_file, 'r') as f:
        content = f.read()
    new_content = content.replace("from s3tokenizer.utils import", "from .utils import")
    with open(target_file, 'w') as f:
        f.write(new_content)
    print("✓ Patched s3tokenizer.py")
else:
    print(f"✗ Could not find {target_file}")

# 4. INSTALL: Install dependencies with proper order to avoid binary incompatibility
print("\n" + "="*60)
print("INSTALLING DEPENDENCIES")
print("="*60)

# Step 1: Install numpy FIRST and ALONE
print("\n[1/5] Installing numpy...")
!pip uninstall -y numpy
!pip install -q 'numpy>=1.24.0,<2.0.0' --only-binary :all:

# Step 2: Reinstall packages that depend on numpy to ensure binary compatibility
print("[2/5] Reinstalling pandas, scipy, pyarrow (numpy dependencies)...")
!pip install --force-reinstall --no-cache-dir -q pandas scipy pyarrow

# Step 3: Install s3tokenizer
print("[3/5] Installing s3tokenizer...")
!pip install -q s3tokenizer

# Step 4: Install chatterbox
print("[4/5] Installing chatterbox...")
!pip install -e . --no-build-isolation

# Step 5: Install additional dependencies
print("[5/5] Installing additional dependencies...")
!pip install -q datasets accelerate wandb soundfile huggingface_hub

print("\n" + "="*60)
print("✓ SETUP COMPLETE!")
print("="*60)
print("\nYou can now proceed to the next cells.")

Cleaning up old files...
/content
Cloning repository...
Cloning into 'chatterbox'...
remote: Enumerating objects: 373, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 373 (delta 101), reused 72 (delta 63), pack-reused 205 (from 2)
Receiving objects: 100% (373/373), 1.57 MiB | 10.08 MiB/s, done.
Resolving deltas: 100% (157/157), done.
/content/chatterbox
Branch 'copilot/add-maltese-language-support' set up to track remote branch 'copilot/add-maltese-language-support' from 'origin'.
Switched to a new branch 'copilot/add-maltese-language-support'
✓ Repository cloned and branch checked out

Patching configuration files...
✓ Patched pyproject.toml
✓ Patched s3tokenizer.py

INSTALLING DEPENDENCIES

[1/5] Installing numpy...
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that 

[5/5] Installing additional dependencies...

✓ SETUP COMPLETE!

You can now proceed to the next cells.


## 2. Configuration

Configure training parameters optimized for Colab free tier.

In [10]:
# Authenticate with Hugging Face
from huggingface_hub import login
from getpass import getpass

# Check if already logged in
try:
    from huggingface_hub import HfFolder
    token = HfFolder.get_token()
    if token:
        print("✓ Already authenticated with Hugging Face")
    else:
        raise ValueError("No token found")
except:
    print("Hugging Face authentication required.")
    print("\nTo get your token:")
    print("1. Go to https://huggingface.co/settings/tokens")
    print("2. Create a new token (read access is sufficient)")
    print("3. Copy and paste it below\n")
    
    hf_token = getpass("Enter your Hugging Face token: ")
    login(token=hf_token)
    print("\n✓ Successfully authenticated with Hugging Face!")
    from huggingface_hub import HfFolder
    HfFolder.save_token(hf_token)
    print("✓ Token saved for future sessions")

✓ Already authenticated with Hugging Face


In [11]:
# Training configuration (optimized for Colab free tier)

CONFIG = {
    # Model and device
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'mixed_precision': True,  # Use FP16 to save memory
    
    # Batch sizes (small to fit in 15GB GPU memory)
    'batch_size': 4,  # Per-device batch size
    'gradient_accumulation_steps': 8,  # Effective batch size: 32
    
    # Optimizer settings
    'learning_rate': 1e-5,  # Conservative to prevent forgetting
    'weight_decay': 0.01,
    'max_grad_norm': 1.0,
    
    # Training steps
    'max_steps': 5000,  # ~2-3 hours on Colab T4
    'warmup_steps': 500,
    'save_steps': 1000,
    'eval_steps': 500,
    'logging_steps': 50,
    
    # Data mixing (prevent forgetting)
    'maltese_ratio': 0.40,  # 40% Maltese
    'arabic_ratio': 0.35,   # 35% Arabic
    'italian_ratio': 0.20,  # 20% Italian
    'english_ratio': 0.05,  # 5% English
    
    # Paths
    'output_dir': './maltese_model_checkpoints',
    'cache_dir': './cache',
    
    # Dataset
    'maltese_dataset': 'Bluefir/MASRI_HEADSET_v2',
    'max_audio_length': 10.0,  # seconds
    'sample_rate': 16000,
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Configuration:
  device: cpu
  mixed_precision: True
  batch_size: 4
  gradient_accumulation_steps: 8
  learning_rate: 1e-05
  weight_decay: 0.01
  max_grad_norm: 1.0
  max_steps: 5000
  warmup_steps: 500
  save_steps: 1000
  eval_steps: 500
  logging_steps: 50
  maltese_ratio: 0.4
  arabic_ratio: 0.35
  italian_ratio: 0.2
  english_ratio: 0.05
  output_dir: ./maltese_model_checkpoints
  cache_dir: ./cache
  maltese_dataset: Bluefir/MASRI_HEADSET_v2
  max_audio_length: 10.0
  sample_rate: 16000


## 3. Load and Prepare Data

Load Maltese dataset and prepare mixed-language training data.

In [12]:
from datasets import load_dataset, concatenate_datasets
import numpy as np

# Load Maltese dataset
print("\nLoading Maltese dataset (MASRI_HEADSET_v2)...")
try:
    maltese_dataset = load_dataset(
        CONFIG['maltese_dataset'],
        cache_dir=CONFIG['cache_dir'],
        split='train'  # Adjust split if needed
    )
    print(f"✓ Maltese dataset loaded: {len(maltese_dataset)} samples")
except Exception as e:
    print(f"✗ Failed to load dataset: {e}")
    print("\nTrying alternative loading method...")
    maltese_dataset = load_dataset(
        CONFIG['maltese_dataset'],
        cache_dir=CONFIG['cache_dir']
    )
    # Get the first available split
    split_name = list(maltese_dataset.keys())[0]
    print(f"Available splits: {list(maltese_dataset.keys())}")
    maltese_dataset = maltese_dataset[split_name]
    print(f"✓ Loaded split '{split_name}' with {len(maltese_dataset)} samples")

# Inspect dataset structure
print(f"\nDataset structure:")
print(f"  Features: {maltese_dataset.features}")
print(f"  Columns: {maltese_dataset.column_names}")

# Show sample statistics - handle different possible column names
duration_cols = ['duration', 'length', 'audio_length']
text_cols = ['normalized_text', 'text', 'sentence', 'transcript']
speaker_cols = ['speaker_id', 'speaker', 'client_id']

# Find the right columns
duration_col = next((c for c in duration_cols if c in maltese_dataset.column_names), None)
text_col = next((c for c in text_cols if c in maltese_dataset.column_names), None)
speaker_col = next((c for c in speaker_cols if c in maltese_dataset.column_names), None)

if duration_col:
    durations = maltese_dataset[duration_col]
    print(f"\nAudio duration statistics (using '{duration_col}'):")
    print(f"  Mean: {np.mean(durations):.2f}s")
    print(f"  Min: {np.min(durations):.2f}s")
    print(f"  Max: {np.max(durations):.2f}s")
else:
    print("\nNo duration column found")

# Show a few text samples
print(f"\nSample texts (using '{text_col}' column):")
for i in range(min(3, len(maltese_dataset))):
    text = maltese_dataset[text_col][i] if text_col else "N/A"
    duration = maltese_dataset[duration_col][i] if duration_col else 0
    speaker = maltese_dataset[speaker_col][i] if speaker_col else "unknown"
    print(f"  {i+1}. [{duration:.1f}s, {speaker}] {text}")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [21]:
# For preventing forgetting, we need samples from other languages
# This is a simplified version - in production, use actual multilingual datasets

print("Note: For full training, you should also load:")
print("  - Arabic dataset (35% of training)")
print("  - Italian dataset (20% of training)")
print("  - English dataset (5% of training)")
print("\nFor this demo, we'll focus on Maltese with the understanding that")
print("mixing with other languages is crucial for preventing forgetting.")
print("\nRecommended datasets:")
print("  - Arabic: Common Voice (ar)")
print("  - Italian: Common Voice (it)")
print("  - English: LJSpeech or Common Voice (en)")

# Prepare Maltese data with language tags
def add_language_tag(example):
    example['language_id'] = 'mt'
    return example

maltese_dataset = maltese_dataset.map(add_language_tag)
print(f"\nMaltese dataset prepared with language tags.")

Note: For full training, you should also load:
  - Arabic dataset (35% of training)
  - Italian dataset (20% of training)
  - English dataset (5% of training)

For this demo, we'll focus on Maltese with the understanding that
mixing with other languages is crucial for preventing forgetting.

Recommended datasets:
  - Arabic: Common Voice (ar)
  - Italian: Common Voice (it)
  - English: LJSpeech or Common Voice (en)


Map:   0%|          | 0/3983 [00:00<?, ? examples/s]


Maltese dataset prepared with language tags.


## 4. Load and Prepare Model

Load pre-trained model and configure for training.

In [48]:
# Import model (using correct package path after installation)
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
import torch

# Load pre-trained model
print("Loading Chatterbox Multilingual TTS model...")
print("(This may take a few minutes)")

model = ChatterboxMultilingualTTS.from_pretrained(device=CONFIG['device'])

print("✓ Model loaded successfully!")
print(f"Device: {model.device}")

/content/chatterbox


PackageNotFoundError: No package metadata was found for chatterbox-tts

In [49]:
# Optional: Update vocabulary to include [mt] token
# This is only needed if you've run the update_vocabulary.py script
# and have the updated vocabulary file

# Uncomment if you have updated vocabulary:
# new_vocab_size = 2455  # 2454 + 1 for [mt]
# model.t3.resize_text_token_embeddings(new_vocab_size)
# print(f"Vocabulary resized to {new_vocab_size}")

print("Note: Using existing vocabulary. [mt] token will be treated as [UNK].")
print("For optimal results, run update_vocabulary.py first.")

Note: Using existing vocabulary. [mt] token will be treated as [UNK].
For optimal results, run update_vocabulary.py first.


In [50]:
# Freeze speech encoder and decoder (language-independent)
print("Configuring trainable parameters...")

# Freeze voice encoder
for param in model.ve.parameters():
    param.requires_grad = False
print("✓ Voice encoder frozen")

# Freeze S3Gen decoder
for param in model.s3gen.parameters():
    param.requires_grad = False
print("✓ S3Gen decoder frozen")

# First, freeze everything in T3
for param in model.t3.parameters():
    param.requires_grad = False

# Then selectively unfreeze text-related components
# These are the components that learn language-specific knowledge
trainable_modules = []

# Text embeddings (critical for new language tokens)
if hasattr(model.t3, 'text_emb'):
    for param in model.t3.text_emb.parameters():
        param.requires_grad = True
    trainable_modules.append('text_emb')

# Text head (output projection for text tokens)
if hasattr(model.t3, 'text_head'):
    for param in model.t3.text_head.parameters():
        param.requires_grad = True
    trainable_modules.append('text_head')

# Transformer backbone (for learning text-to-speech mapping)
if hasattr(model.t3, 'tfmr'):
    for param in model.t3.tfmr.parameters():
        param.requires_grad = True
    trainable_modules.append('tfmr')

# Conditioning encoder
if hasattr(model.t3, 'cond_enc'):
    for param in model.t3.cond_enc.parameters():
        param.requires_grad = True
    trainable_modules.append('cond_enc')

print(f"✓ Trainable T3 modules: {trainable_modules}")

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.t3.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model.t3.parameters() if not p.requires_grad)
total_t3_params = sum(p.numel() for p in model.t3.parameters())
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad is not None)

print(f"\nT3 Model:")
print(f"  Trainable: {trainable_params / 1e6:.2f}M")
print(f"  Frozen: {frozen_params / 1e6:.2f}M")
print(f"  Total: {total_t3_params / 1e6:.2f}M")
print(f"\nTraining {trainable_params / total_t3_params * 100:.1f}% of T3 model")

Configuring trainable parameters...


NameError: name 'model' is not defined

In [51]:
# Enable mixed precision training (saves memory)
if CONFIG['mixed_precision'] and CONFIG['device'] == 'cuda':
    # Handle different PyTorch versions
    try:
        # PyTorch 2.0+
        from torch.amp import GradScaler
        scaler = GradScaler('cuda')
    except (ImportError, TypeError):
        # Older PyTorch
        from torch.cuda.amp import GradScaler
        scaler = GradScaler()
    print("✓ Mixed precision (FP16) enabled")
else:
    scaler = None
    print("Running in FP32 mode")

Running in FP32 mode


## 5. Training Loop

Simplified training loop optimized for Colab.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Setup optimizer
optimizer = AdamW(
    [p for p in model.t3.parameters() if p.requires_grad],
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
    betas=(0.9, 0.999)
)

# Learning rate scheduler
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['max_steps']
)

print("✓ Optimizer configured")
print(f"  Learning rate: {CONFIG['learning_rate']}")
print(f"  Weight decay: {CONFIG['weight_decay']}")

In [ ]:
import os
from tqdm.auto import tqdm

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print("="*60)
print("IMPORTANT NOTES FOR FULL TRAINING")
print("="*60)
print("\nThis notebook demonstrates the training structure, but for")
print("production use, you MUST implement:")
print("\n1. DATA PREPROCESSING:")
print("   - Audio loading and preprocessing")
print("   - Speech tokenization with S3Tokenizer")
print("   - Text tokenization with MTLTokenizer")
print("   - Proper batching and padding")
print("\n2. MIXED-LANGUAGE SAMPLING:")
print("   - Load Arabic, Italian, English datasets")
print("   - Sample according to CONFIG ratios")
print("   - This prevents catastrophic forgetting!")
print("\n3. VALIDATION:")
print("   - Evaluate on Maltese test set")
print("   - Monitor Arabic/Italian performance (should not drop >5%)")
print("   - Check for catastrophic forgetting")
print("\n4. CHECKPOINTING:")
print("   - Save model every CONFIG['save_steps']")
print("   - Keep best checkpoint based on validation")
print("="*60)

# Simplified training loop structure
print("\nTraining loop structure (REQUIRES DATA PIPELINE IMPLEMENTATION):")
print("""\n
model.t3.train()
for step in range(CONFIG['max_steps']):
    # 1. Sample batch (mixed languages according to CONFIG ratios)
    # batch = sample_mixed_batch(maltese_loader, arabic_loader, italian_loader, english_loader)
    
    # 2. Forward pass
    # loss_text, loss_speech = model.t3.loss(
    #     t3_cond=batch['t3_cond'],
    #     text_tokens=batch['text_tokens'],
    #     text_token_lens=batch['text_token_lens'],
    #     speech_tokens=batch['speech_tokens'],
    #     speech_token_lens=batch['speech_token_lens']
    # )
    # loss = loss_text + loss_speech
    
    # 3. Backward pass with gradient accumulation
    # if scaler:
    #     scaler.scale(loss).backward()
    # else:
    #     loss.backward()
    
    # 4. Optimizer step (after accumulation)
    # if (step + 1) % CONFIG['gradient_accumulation_steps'] == 0:
    #     if scaler:
    #         scaler.unscale_(optimizer)
    #         torch.nn.utils.clip_grad_norm_(model.t3.parameters(), CONFIG['max_grad_norm'])
    #         scaler.step(optimizer)
    #         scaler.update()
    #     else:
    #         torch.nn.utils.clip_grad_norm_(model.t3.parameters(), CONFIG['max_grad_norm'])
    #         optimizer.step()
    #     scheduler.step()
    #     optimizer.zero_grad()
    
    # 5. Logging and checkpointing
    # if step % CONFIG['logging_steps'] == 0:
    #     print(f"Step {step}: Loss={loss.item():.4f}, LR={scheduler.get_last_lr()[0]:.2e}")
    # if step % CONFIG['save_steps'] == 0:
    #     save_checkpoint(step)
""")

print("\nFor a complete implementation, see train_maltese.py")
print("and MALTESE_FINETUNING_GUIDE.md in the repository.")

## 6. Example: Process Single Maltese Sample

Demonstrate how to process a single training sample.

In [ ]:
# Example of processing a single Maltese sample
import librosa

def process_maltese_sample(audio_path, text, model):
    """
    Example function showing how to process a single training sample.
    
    In production, this should be batched and optimized.
    """
    # 1. Load and preprocess audio
    audio, sr = librosa.load(audio_path, sr=16000)
    
    # 2. Tokenize text
    text_tokens = model.tokenizer.text_to_tokens(text, language_id='mt')
    
    # 3. Extract speech tokens (requires S3Tokenizer)
    # speech_tokens = model.s3gen.tokenizer.encode(audio)
    
    # 4. Prepare conditioning (speaker embedding from audio)
    # speaker_emb = model.ve.embeds_from_wavs([audio], sample_rate=16000)
    
    return text_tokens

# Demo with first sample - use the text column we found earlier
if len(maltese_dataset) > 0:
    sample = maltese_dataset[0]
    
    # Try different possible text column names
    text = None
    for col in ['normalized_text', 'text', 'sentence', 'transcript']:
        if col in sample:
            text = sample[col]
            break
    
    if text is None:
        text = 'Bonġu! Kif int illum?'
        print("Using fallback text (no text column found)")
    
    print(f"Sample text: {text}")
    try:
        text_tokens = model.tokenizer.text_to_tokens(text, language_id='mt')
        print(f"Text tokens shape: {text_tokens.shape}")
        print(f"First 10 tokens: {text_tokens[0, :10].tolist()}")
    except Exception as e:
        print(f"Tokenization error: {e}")
        print("This may be expected if Maltese [mt] token is not in vocabulary")

## 7. Save Model

Save trained model checkpoint.

In [ ]:
def save_checkpoint(model, step, output_dir):
    """
    Save model checkpoint.
    """
    checkpoint_dir = os.path.join(output_dir, f'checkpoint-{step}')
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    # Save T3 model (only trainable part)
    checkpoint_path = os.path.join(checkpoint_dir, 't3_maltese.pt')
    torch.save({
        'step': step,
        'model_state_dict': model.t3.state_dict(),
        'text_tokens_dict_size': model.t3.hp.text_tokens_dict_size,
    }, checkpoint_path)
    
    print(f"✓ Checkpoint saved to {checkpoint_path}")
    return checkpoint_path

# Save initial checkpoint (before training)
initial_checkpoint = save_checkpoint(model, 0, CONFIG['output_dir'])
print(f"\nInitial checkpoint: {initial_checkpoint}")

## 8. Test Generation

Test Maltese speech generation with the model.

In [ ]:
# Test generation
import torchaudio
from IPython.display import Audio, display
import tempfile
import soundfile as sf

# Test Maltese text
test_texts = [
    "Bonġu! Kif int illum?",
    "Malta għandha storja kbira.",
    "Jiena kuntent li niltaqa' miegħek."
]

print("Testing Maltese generation...\n")

# Check if we need to prepare a voice prompt from the dataset
audio_prompt_path = None
if len(maltese_dataset) > 0 and 'audio' in maltese_dataset.column_names:
    # Extract audio from first sample as reference
    sample = maltese_dataset[0]
    audio_data = sample['audio']
    
    # Save to temp file
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
        audio_prompt_path = f.name
        # Handle different audio formats from datasets
        if isinstance(audio_data, dict):
            sf.write(audio_prompt_path, audio_data['array'], audio_data['sampling_rate'])
        else:
            print("Audio format not recognized, using built-in voice")
            audio_prompt_path = None
    
    if audio_prompt_path:
        print(f"Using voice prompt from dataset: {audio_prompt_path}\n")

# Check if model has pre-loaded conditionals
has_conds = model.conds is not None
if not has_conds and audio_prompt_path is None:
    print("⚠ No voice prompt available and no built-in conditionals.")
    print("Generation requires an audio prompt to clone the voice from.")
    print("Skipping generation test - load an audio file to test.\n")
else:
    for i, text in enumerate(test_texts):
        print(f"Text {i+1}: {text}")
        
        try:
            # Generate audio
            with torch.no_grad():
                wav = model.generate(
                    text, 
                    language_id='mt',
                    audio_prompt_path=audio_prompt_path,
                    exaggeration=0.5,
                    cfg_weight=0.5,
                    temperature=0.8
                )
            
            # Save and play
            output_path = f"maltese_test_{i+1}.wav"
            torchaudio.save(output_path, wav.cpu(), model.sr)
            print(f"✓ Saved to {output_path}")
            
            # Play in notebook
            display(Audio(wav.squeeze().cpu().numpy(), rate=model.sr))
        except Exception as e:
            print(f"✗ Error: {e}")
            import traceback
            traceback.print_exc()
        
        print()

# Cleanup temp file
if audio_prompt_path:
    import os
    try:
        os.unlink(audio_prompt_path)
    except:
        pass

## 9. Next Steps

To complete the full training pipeline:

### Required Implementations:

1. **Data Pipeline**:
   ```python
   - Implement audio loading and preprocessing
   - Extract speech tokens with S3Tokenizer
   - Create proper DataLoader with batching
   - Handle variable-length sequences
   ```

2. **Mixed-Language Sampling**:
   ```python
   - Load additional datasets (Arabic, Italian, English)
   - Implement sampling according to CONFIG ratios
   - Ensure balanced batches across languages
   ```

3. **Validation Loop**:
   ```python
   - Create validation dataset
   - Monitor loss on all languages
   - Check for catastrophic forgetting
   - Save best checkpoint
   ```

4. **Full Training**:
   ```python
   - Run for CONFIG['max_steps'] (5000 steps)
   - Monitor training loss
   - Validate every 500 steps
   - Save checkpoints every 1000 steps
   ```

### Resources:

- **MALTESE_FINETUNING_GUIDE.md**: Detailed training guide
- **train_maltese.py**: Complete training script template
- **VOCABULARY_UPDATE_GUIDE.md**: Update vocabulary with [mt] token

### Monitoring:

Track these metrics to ensure quality:
- **Maltese loss**: Should decrease steadily
- **Arabic/Italian performance**: Should stay within 95% of baseline
- **Memory usage**: Should stay under 15GB for Colab free tier
- **Training time**: ~2-3 hours for 5000 steps on T4 GPU

### Preventing Catastrophic Forgetting:

Critical strategies implemented:
1. ✅ **Conservative learning rate** (1e-5)
2. ✅ **Mixed-language training** (40% mt, 35% ar, 20% it, 5% en)
3. ✅ **Smart initialization** (embeddings from mean/std)
4. ✅ **Gradient clipping** (max_norm=1.0)
5. ✅ **Regular validation** (check all languages)


## Additional Resources

- **Dataset**: [Bluefir/MASRI_HEADSET_v2](https://huggingface.co/datasets/Bluefir/MASRI_HEADSET_v2)
- **Repository**: [Wubpooz/chatterbox](https://github.com/Wubpooz/chatterbox)
- **Branch**: `copilot/add-maltese-language-support`
- **Documentation**:
  - MALTESE_FINETUNING_GUIDE.md
  - MALTESE_IMPLEMENTATION.md
  - VOCABULARY_UPDATE_GUIDE.md
